# DistilBERT Intent Classifier — Evaluation Analysis
### Sunlytics CRS — M3 Adaptive RAG | 8-Class Intent Classification

Loads the trained model from Google Drive and generates full evaluation analysis.  
**No retraining needed — run all cells top to bottom.**

---
**Files needed in `MyDrive/sunlytics_eval/`:**

| File | Source |
|------|--------|
| `model.safetensors` | `outputs/old_three/best_model/` |
| `config.json` | `outputs/old_three/best_model/` |
| `tokenizer.json` | `outputs/old_three/best_model/` |
| `tokenizer_config.json` | `outputs/old_three/best_model/` |
| `v3_test_realistic.csv` | `data/v3_test_realistic.csv` |

**Runtime:** T4 GPU → `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── CELL 1 — Install + GPU check ─────────────────────────────────────────────
!pip install 'transformers>=4.40.0' scikit-learn pandas seaborn matplotlib -q

import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device :', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── CELL 2 — Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_FOLDER = '/content/drive/MyDrive/sunlytics_eval'
SAVE_DIR     = '/content/drive/MyDrive/sunlytics_eval/intent_eval_results'
os.makedirs(SAVE_DIR, exist_ok=True)

# Verify required files
required = ['model.safetensors', 'config.json', 'tokenizer.json',
            'tokenizer_config.json', 'v3_test_realistic.csv']
print('Checking files in Drive folder:')
all_ok = True
for f in required:
    path   = os.path.join(DRIVE_FOLDER, f)
    exists = os.path.isfile(path)
    size   = f'{os.path.getsize(path)/1e6:.1f} MB' if exists else 'MISSING'
    status = '✓' if exists else '✗'
    print(f'  {status} {f:40s} {size}')
    if not exists: all_ok = False

print()
print('All files present — ready to proceed.' if all_ok else 'Upload missing files to Drive before continuing.')

In [ ]:
# ── CELL 3 — Config ───────────────────────────────────────────────────────────
import random
import numpy as np

SEED       = 42
MAX_LEN    = 256
BATCH_SIZE = 64

LABEL_NAMES = [
    'INITIAL_REQUEST',
    'REFINEMENT',
    'ATTRIBUTE_QUESTION',
    'EXPLANATION_WHY',
    'COMPARISON',
    'SELECTION_REFERENCE',
    'FEEDBACK',
    'CHITCHAT',
]

RETRIEVAL_STRATEGY = {
    'INITIAL_REQUEST':    'FULL',
    'REFINEMENT':         'FULL',
    'ATTRIBUTE_QUESTION': 'PARTIAL',
    'EXPLANATION_WHY':    'PARTIAL',
    'COMPARISON':         'PARTIAL',
    'SELECTION_REFERENCE':'PARTIAL',
    'FEEDBACK':           'NO',
    'CHITCHAT':           'NO',
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# ── CELL 4 — Load model + tokenizer + test set ────────────────────────────────
import pandas as pd
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer
from torch.utils.data import Dataset, DataLoader

print('Loading tokenizer...')
tokenizer = DistilBertTokenizer.from_pretrained(DRIVE_FOLDER)
print('Loading model...')
model = DistilBertForSequenceClassification.from_pretrained(DRIVE_FOLDER)
model.to(device)
model.eval()
print(f'Model loaded — {sum(p.numel() for p in model.parameters()):,} parameters')

df = pd.read_csv(os.path.join(DRIVE_FOLDER, 'v3_test_realistic.csv'))
print(f'\nTest set loaded: {len(df):,} rows')
print('Label distribution:')
for i, name in enumerate(LABEL_NAMES):
    n = (df['label'] == i).sum()
    print(f'  {i}  {name:<25} {n}')


class IntentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = [str(t) for t in texts]
        self.labels = list(labels)
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True, padding='max_length',
                        max_length=MAX_LEN, return_tensors='pt')
        return {'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'label':          torch.tensor(self.labels[idx], dtype=torch.long)}


def run_inference(texts, labels):
    ds = IntentDataset(texts, labels)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    preds, trues = [], []
    with torch.no_grad():
        for batch in dl:
            out = model(batch['input_ids'].to(device),
                        batch['attention_mask'].to(device))
            preds.extend(torch.argmax(out.logits, 1).cpu().numpy())
            trues.extend(batch['label'].numpy())
    return np.array(preds), np.array(trues)


print('\nRunning inference on test set...')
y_pred, y_true = run_inference(df['input_text'].tolist(), df['label'].tolist())
print('Inference complete.')

In [ ]:
# ── CELL 5 — Print metrics summary ────────────────────────────────────────────
from sklearn.metrics import classification_report, accuracy_score, f1_score

acc     = accuracy_score(y_true, y_pred)
mac_f1  = f1_score(y_true, y_pred, average='macro')
wgt_f1  = f1_score(y_true, y_pred, average='weighted')
pcf1    = f1_score(y_true, y_pred, average=None)

print('=' * 60)
print('  DistilBERT 8-Class Intent Classifier — Test Results')
print('  Dataset: v3_test_realistic.csv')
print('=' * 60)
print(f'  Accuracy    : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Macro-F1    : {mac_f1:.4f}')
print(f'  Weighted-F1 : {wgt_f1:.4f}')
print()
print('  Per-class F1:')
for name, v in zip(LABEL_NAMES, pcf1):
    bar = '█' * int(v * 30)
    print(f'  {name:<25} {v:.4f}  {bar}')
print()
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

In [ ]:
# ── CELL 6 — VISUALISATION 1: Confusion Matrix ────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

short = ['INIT', 'REFINE', 'ATTR_Q', 'WHY', 'COMP', 'SELECT', 'FDBK', 'CHAT']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Confusion Matrix — DistilBERT 8-Class Intent Classifier\n'
             f'Test set: v3_test_realistic.csv ({len(y_true):,} samples)  |  '
             f'Accuracy: {acc*100:.2f}%  |  Macro-F1: {mac_f1:.4f}',
             fontsize=13, fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short, yticklabels=LABEL_NAMES,
            ax=axes[0], linewidths=0.4, linecolor='lightgray',
            annot_kws={'size': 10})
axes[0].set_title('Raw Counts', fontsize=12)
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('True', fontsize=11)
axes[0].tick_params(axis='x', rotation=45)

# Row-normalised
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=short, yticklabels=LABEL_NAMES,
            ax=axes[1], linewidths=0.4, linecolor='lightgray',
            vmin=0.0, vmax=1.0, annot_kws={'size': 10})
axes[1].set_title('Row-normalised (recall per class)', fontsize=12)
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_ylabel('True', fontsize=11)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

In [ ]:
# ── CELL 7 — VISUALISATION 2: Per-class F1 bar chart ─────────────────────────
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np

report = classification_report(y_true, y_pred,
                                target_names=LABEL_NAMES, output_dict=True)

metrics_list = ['precision', 'recall', 'f1-score']
x      = np.arange(len(LABEL_NAMES))
width  = 0.26
colors = ['#4C72B0', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(14, 6))
for i, (metric, color) in enumerate(zip(metrics_list, colors)):
    values = [report[lbl][metric] for lbl in LABEL_NAMES]
    bars = ax.bar(x + i * width, values, width,
                  label=metric.capitalize(), color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')

ax.axhline(y=mac_f1, color='black', linestyle='--', linewidth=1.5,
           label=f'Macro F1 = {mac_f1:.4f}')
ax.set_xlabel('Intent Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-class Precision / Recall / F1 — DistilBERT Intent Classifier',
             fontsize=12, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(LABEL_NAMES, rotation=30, ha='right', fontsize=10)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: per_class_metrics.png')

In [ ]:
# ── CELL 8 — VISUALISATION 3: Retrieval Strategy Accuracy ────────────────────
# Groups the 8 classes into FULL / PARTIAL / NO and shows accuracy per group.
import matplotlib.pyplot as plt
import numpy as np

strategy_groups = {
    'FULL\n(INITIAL_REQUEST\n+ REFINEMENT)':    [0, 1],
    'PARTIAL\n(ATTR_Q, WHY\nCOMP, SELECT)':     [2, 3, 4, 5],
    'NO\n(FEEDBACK\n+ CHITCHAT)':               [6, 7],
}

strategy_acc, strategy_labels, strategy_counts = [], [], []
for strat_name, class_ids in strategy_groups.items():
    mask   = np.isin(y_true, class_ids)
    if mask.sum() == 0:
        continue
    s_acc  = accuracy_score(y_true[mask], y_pred[mask])
    strategy_acc.append(s_acc)
    strategy_labels.append(strat_name)
    strategy_counts.append(mask.sum())

colors = ['#2196F3', '#FF9800', '#4CAF50']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Retrieval Strategy Group Analysis — DistilBERT Intent Classifier',
             fontsize=13, fontweight='bold')

# Accuracy bar chart
bars = axes[0].bar(strategy_labels, strategy_acc,
                   color=colors, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, strategy_acc):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                 f'{val*100:.1f}%', ha='center', va='bottom',
                 fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 1.12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Accuracy per Retrieval Strategy Group', fontsize=11)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)
axes[0].axhline(y=acc, color='black', linestyle='--', linewidth=1,
                label=f'Overall accuracy ({acc*100:.1f}%)')
axes[0].legend(fontsize=9)

# Sample count pie chart
axes[1].pie(strategy_counts, labels=strategy_labels, colors=colors,
            autopct='%1.0f%%', startangle=90,
            textprops={'fontsize': 10})
axes[1].set_title('Test Set Distribution by Retrieval Group', fontsize=11)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/retrieval_strategy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: retrieval_strategy_analysis.png')

In [ ]:
# ── CELL 9 — VISUALISATION 4: Most Confused Class Pairs ──────────────────────
import matplotlib.pyplot as plt
import pandas as pd

# Find off-diagonal confusion pairs
confused_pairs = []
for i in range(len(LABEL_NAMES)):
    for j in range(len(LABEL_NAMES)):
        if i != j and cm[i, j] > 0:
            confused_pairs.append({
                'True':      LABEL_NAMES[i],
                'Predicted': LABEL_NAMES[j],
                'Count':     cm[i, j],
                'Rate':      round(cm_norm[i, j], 3),
            })

pairs_df = pd.DataFrame(confused_pairs).sort_values('Count', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
bar_labels = [f"{r['True']}\n→ {r['Predicted']}" for _, r in pairs_df.iterrows()]
bars = ax.barh(bar_labels, pairs_df['Count'], color='#E57373', alpha=0.85, edgecolor='white')

for bar, (_, row) in zip(bars, pairs_df.iterrows()):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{row['Count']} ({row['Rate']*100:.1f}%)",
            va='center', fontsize=9)

ax.set_xlabel('Misclassification Count', fontsize=11)
ax.set_title('Top 10 Most Confused Class Pairs', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/confused_pairs.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top confused pairs:')
print(pairs_df.to_string(index=False))
print('Saved: confused_pairs.png')

In [ ]:
# ── CELL 10 — Save metrics JSON + download all ────────────────────────────────
import json
import shutil
from google.colab import files

eval_summary = {
    'model':        'DistilBERT fine-tuned on v4_train_balanced (52,028 samples)',
    'test_set':     'v3_test_realistic.csv (1,787 samples)',
    'accuracy':     round(float(acc),     4),
    'macro_f1':     round(float(mac_f1),  4),
    'weighted_f1':  round(float(wgt_f1),  4),
    'per_class_f1': {name: round(float(v), 4)
                     for name, v in zip(LABEL_NAMES, pcf1)},
    'retrieval_strategy_accuracy': {
        strat: round(float(strategy_acc[i]), 4)
        for i, strat in enumerate(['FULL', 'PARTIAL', 'NO'])
    },
}

with open(f'{SAVE_DIR}/eval_summary.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

with open(f'{SAVE_DIR}/classification_report.txt', 'w') as f:
    f.write('DistilBERT 8-Class Intent Classifier — Evaluation Report\n')
    f.write(f'Test set : v3_test_realistic.csv ({len(y_true)} samples)\n')
    f.write(f'Accuracy : {acc*100:.2f}%\n')
    f.write(f'Macro-F1 : {mac_f1:.4f}\n')
    f.write('=' * 60 + '\n\n')
    f.write(classification_report(y_true, y_pred,
                                   target_names=LABEL_NAMES, digits=4))

print('Files saved to Drive:', SAVE_DIR)
for fn in sorted(os.listdir(SAVE_DIR)):
    print(f'  {fn}')

# Download as ZIP
shutil.make_archive('/content/intent_eval_results', 'zip', SAVE_DIR)
files.download('/content/intent_eval_results.zip')
print('\nDownload started.')